In [1]:
# -*- coding: utf-8 -*-
"""
동영상 시각 → 로그 타임스탬프 변환 후, relationship_node.created 이벤트를 자동 삽입.

사용법:
  python insert_relationship_events.py

필수: 같은 디렉토리에 P1.log, P2.log ... 가 존재.
결과:
  - P#.log.patched : 삽입된 이벤트 포함, 시간순 정렬
  - inserted_relationship_events_summary.csv : 삽입 요약
"""

import re
import json
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional, Tuple
# 파일 상단 근처에 추가(표준 라이브러리만 사용)
import difflib

def _norm(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def _tokens(s: str) -> set:
    # 영문/숫자 토큰 기준(라벨들 대부분 영어라 가볍게 처리)
    return set(re.findall(r"[a-z0-9]+", _norm(s)))

def find_instance_id_by_label(idx: List[Tuple[datetime, str, str]],
                              needle: str,
                              prefer_around: Optional[datetime] = None) -> Optional[str]:
    """
    간단·견고한 탐색:
      1) 대소문자 무시 부분일치
      2) 토큰(단어) 부분집합 매칭
      3) difflib 유사도
      4) prefer_around(이벤트 시각) 근처에 최근성 가중(±30분)
    """
    n_norm = _norm(needle)
    n_tok  = _tokens(needle)

    best_iid = None
    best_score = -1.0

    for ts, iid, lab in idx:
        l_norm = _norm(lab)
        l_tok  = _tokens(lab)

        score = 0.0

        # (1) 부분일치
        if n_norm and n_norm in l_norm:
            score += 2.0

        # (2) 토큰 부분집합(needle의 단어들이 전부 들어가면 가산점)
        if n_tok and n_tok.issubset(l_tok):
            score += 1.5

        # (3) difflib 유사도 (0~1)
        score += 1.0 * difflib.SequenceMatcher(None, n_norm, l_norm).ratio()

        # (4) 최근성 가중(선택)
        if prefer_around:
            dt = abs((prefer_around - ts).total_seconds())
            if dt <= 30 * 60:         # 30분 이내
                score += 0.4
            elif dt <= 2 * 3600:      # 2시간 이내
                score += 0.15

        if score > best_score:
            best_score = score
            best_iid = iid

    # 아주 낮은 스코어(=사실상 불일치)면 None 처리
    # (라벨이 완전히 다른데 억지로 붙지 않도록 약한 기준선)
    if best_score < 0.9:   # 필요하면 0.8~1.0 사이에서 조정
        return None
    return best_iid

# ─────────────────────────────────────────────────────────────────────────────
# 0) 입력 데이터: 사용자 제공 앵커/이벤트 목록
#    - 각 P# 블록의 첫 줄은 "앵커(동영상 시각) : 앵커 로그 타임스탬프"
#    - 이어지는 줄들은 "이벤트 동영상 시각: src-dst, relation" 또는 변형 포맷
#    - 아래에 사용자가 준 값을 구조화해 두었음.
# ─────────────────────────────────────────────────────────────────────────────

CONFIG = {
    "P1": {
        "anchor_video_time": "0:39",
        "anchor_log_ts": "2025-08-22T08:33:16.620253",
        "events": [
            ("4:41", "rabbit", "balloon", "cuddling"),
            ("4:54", "Rabbit", "balloon", "talking to"),
            ("27:06", "Baby rabbit", "rabbit", "talking to"),
            ("27:16", "Baby rabbit", "rabbit", "talking to"),
            ("27:27", "rabbit", "Baby rabbit", "watching"),
            ("37:44", "Baby rabbit", "tree", "looking/pointing at"),
            ("38:40", "Baby rabbit", "tree", "looking up"),
            ("39:13", "rabbit", "tree", "looking up"),
            ("39:16", "Balloon", "tree", "stuck in"),
            ("44:47", "rabbit", "Balloon", "grabbing"),
            ("45:33", "rabbit", "Baby rabbit", "hugging together"),
            ("45:43", "Baby rabbit", "Baby rabbit", "hugging together"),
            ("45:48", "Baby rabbit", "rabbit", "hugging together"),
        ],
    },
    "P3": {
        "anchor_video_time": "19:45",
        "anchor_log_ts": "2025-08-23T06:15:43.813391",
        "events": [
            ("34:21", "Rabbit", "balloon", "giving"),
            ("34:28", "rabbit", "balloon", "receiving"),
            ("44:40", "rabbit", "balloon", "looking up"),
            ("57:21", "Rabbit", "rabbit", "console"),
            ("57:35", "Rabbit", "rabbit", "console"),
            ("1:05:49", "rabbit", "balloon", "poing"),
        ],
    },
    "P4": {
        "anchor_video_time": "26:04",
        "anchor_log_ts": "2025-08-25T12:36:42.993017",
        "events": [
            ("44:08", "Bunny", "tree", "looking up"),
            ("44:18", "Bunny", "tree", "looking up"),
            ("44:37", "bunny", "tree", "looking up"),
            ("48:03", "Bunny", "tree", "hanging on the highest branch of the tree"),
            ("48:52", "Bunny", "bunny", "looking down at"),
            ("50:55", "Bunny", "balloon", "holding"),
            ("51:12", "Bunny", "Bunny", "looking down at"),
            ("1:01:56", "bunny", "balloon", "looking at"),
        ],
    },
    "P5": {
        "anchor_video_time": "10:50",
        "anchor_log_ts": "2025-08-29T23:59:30.158284",
        "events": [
            ("43:49", "tree", "balloon", "related"),
            ("44:20", "rabbit", "tree", "looking at"),
            ("44:26", "rabbit", "tree", "pointing at"),
            ("44:44", "rabbit", "tree", "looking at"),
            ("49:01", "tree", "rabbit", "on the"),
            ("49:20", "tree", "balloon", "on the"),
            ("49:26", "rabbit", "balloon", "looking at"),
            ("49:43", "rabbit", "balloon", "looking at"),
            ("53:38", "rabbit", "balloon", "holding"),
        ],
    },
    "P6": {
        "anchor_video_time": "11:36",
        "anchor_log_ts": "2025-08-30T01:51:32.630419",
        "events": [
            ("34:39", "rabbit", "balloon", "looking"),
            ("34:38", "rabbit", "balloon", "looking"),
            ("34:40", "Rabbit", "balloon", "looking"),
            ("37:08", "tree", "balloon", "holding"),
            ("41:01", "Rabbit", "tree", "on"),
            ("47:05", "rabbit", "rabbit", "holding hands"),
            ("48:01", "rabbit", "rabbit", "holding hands"),
        ],
    },
}


# CONFIG = {
#     "P1": {
#         "anchor_video_time": "0:39",  # 39초
#         "anchor_log_ts": "2025-08-24T11:28:00.507241",
#         "events": [
#             ("26:32", "woman", "man", "holding"),
#             ("34:02", "grandpa", "girl", "watching"),
#             ("34:40", "girl", "balloon", "holding"),
#         ],
#     },
#     "P2": {
#         "anchor_video_time": "6:48",
#         "anchor_log_ts": "2025-08-14T01:16:33.917189",
#         "events": [
#             ("10:38", "Female", "Man", "holding hands"),
#             ("11:16", "Female", "basket", "holding"),
#             ("11:23", "Man", "basket", "holding"),
#             ("12:20", "Man", "pan", "holding up with one hand"),
#             ("12:56", "bottle", "pan", "on"),
#             ("14:18", "Child", "ballon", "holding"),
#             ("14:48", "Child", "bear", "holding"),
#         ],
#     },
#     "P4": {
#         "anchor_video_time": "18:03",
#         "anchor_log_ts": "2025-08-14T13:39:43.770917",
#         "events": [
#             ("1:11:40", "man", "Woman", "look"),
#             ("1:11:49", "woman", "Woman", "look"),
#             ("1:11:56", "Young man", "Woman", "look"),
#         ],
#     },
#     "P5": {
#         "anchor_video_time": "1:16",
#         "anchor_log_ts": "2025-08-19T07:10:36.345217",
#         "events": [
#             ("24:44", "Female person", "camera", "holding with right hand"),
#             ("25:30", "Female person", "hat", "holding with left hand"),
#             ("26:51", "Female", "luggage", "holding with right hand"),
#             ("27:28", "Female", "coffee", "holding with left hand"),
#         ],
#     },
#     "P6": {
#         "anchor_video_time": "2:12",
#         "anchor_log_ts": "2025-08-20T05:31:10.031337",
#         "events": [
#             ("8:30", "man", "basket", "holding"),
#             ("8:33", "man", "bottle", "holding"),
#             ("9:00", "girl", "Balloon", "holding"),
#             ("15:30", "man", "tray", "holding"),
#         ],
#     },
#     "P7": {
#         "anchor_video_time": "21:52",
#         "anchor_log_ts": "2025-08-20T11:08:00.204696",
#         "events": [
#             ("37:45", "Man", "box", "holding"),
#             ("38:16", "Woman", "box", "holding"),
#             ("38:32", "man", "Girl", "looking at"),
#         ],
#     },
#     "P8": {
#         "anchor_video_time": "2:22",
#         "anchor_log_ts": "2025-08-21T01:56:36.647029",
#         "events": [
#             ("13:05", "man", "woman", "looking at"),
#             ("16:10", "Man", "woman", "looking at"),
#             ("20:15", "Woman", "woman", "looking at"),
#         ],
#     },
#     "P9": {
#         "anchor_video_time": "16:55",
#         "anchor_log_ts": "2025-08-22T13:25:35.646607",
#         "events": [
#             # 사용자가 후속 이벤트를 아직 안 준 상태로 보임 → 필요시 추가
#         ],
#     },
#     "P10": {
#         "anchor_video_time": "4:05",
#         "anchor_log_ts": "2025-08-21T09:50:53.065615",
#         "events": [
#             ("20:40", "Father", "basket", "holding"),
#         ],
#     },
#     "P12": {
#         "anchor_video_time": "15:00",
#         "anchor_log_ts": "2025-08-22T01:34:51.331618",
#         "events": [
#             ("30:30", "ballon", "Girl", "girl is holding balloon in she’s left hand"),
#             ("31:31", "bear", "Girl", "holding teddy on she’s right hand"),
#             ("32:45", "tray", "Man", "put on his left hand"),
#             ("34:00", "basket", "Woman", "holding basket with the man"),
#             ("34:20", "basket", "male", "holding basket with the woman"),
#         ],
#     },
# }

# ─────────────────────────────────────────────────────────────────────────────
# 1) 유틸: 시간 파서
# ─────────────────────────────────────────────────────────────────────────────

def parse_video_time_to_seconds(s: str) -> int:
    """
    허용 포맷:
      - "SS"
      - "M:SS"
      - "MM:SS"
      - "H:MM:SS"
      - "M.SS" / "MM.SS" / "H.MM.SS" (점 구분도 지원)
    """
    s = s.strip()
    s = s.replace(" ", "")
    s = s.replace("m", ":").replace("min", ":")
    s = s.replace("분", ":").replace("초", "")
    # 점 구분을 콜론으로 치환
    s = s.replace(".", ":")
    parts = s.split(":")
    parts = [p for p in parts if p != ""]
    nums = list(map(int, parts))
    if len(nums) == 1:
        return nums[0]
    elif len(nums) == 2:
        m, sec = nums
        return m * 60 + sec
    elif len(nums) == 3:
        h, m, sec = nums
        return h * 3600 + m * 60 + sec
    else:
        raise ValueError(f"Unsupported time format: {s}")

def parse_iso_ts(ts: str) -> datetime:
    # 다양한 ISO 포맷 수용
    fmts = [
        "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%dT%H:%M:%S",
    ]
    for fmt in fmts:
        try:
            return datetime.strptime(ts, fmt)
        except:
            pass
    # pandas에 의존하지 않고 간단 처리
    raise ValueError(f"Unrecognized timestamp: {ts}")

def to_log_prefix(dt: datetime) -> str:
    # "YYYY-MM-DD HH:MM:SS,mmm"
    return dt.strftime("%Y-%m-%d %H:%M:%S,") + f"{int(dt.microsecond/1000):03d}"

def to_iso(dt: datetime) -> str:
    # JSON 내부 timestamp 포맷
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 로그 파싱 & 인스턴스 색인
# ─────────────────────────────────────────────────────────────────────────────

def load_log_lines(log_path: Path) -> List[str]:
    return log_path.read_text(encoding="utf-8", errors="ignore").splitlines()

def extract_json_from_line(line: str) -> Optional[Dict[str, Any]]:
    # " - INFO - " 이후 JSON
    if " - INFO - " not in line:
        return None
    try:
        json_str = line.split(" - INFO - ", 1)[-1].strip()
        return json.loads(json_str)
    except Exception:
        return None

def parse_line_time(line: str) -> Optional[datetime]:
    # 선두의 "YYYY-MM-DD HH:MM:SS,mmm"
    m = re.match(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}),(\d{3})", line)
    if not m:
        return None
    base = datetime.strptime(m.group(1), "%Y-%m-%d %H:%M:%S")
    ms = int(m.group(2))
    return base.replace(microsecond=ms * 1000)

_ID_KEYS = r"(?:instance_id|instanceId|sharedId|nodeId|id)"
_LABEL_KEYS = r"(?:instance_label|instanceLabel|label|instanceName|textDescription)"

def extract_id_label_pairs_by_regex(line: str):
    pairs = []

    # 0) 일반 id/label (기존)
    pat_both = re.compile(
        rf'"{_ID_KEYS}"\s*:\s*"([^"]+)"'
        rf'.{{0,400}}?'
        rf'"{_LABEL_KEYS}"\s*:\s*"([^"]+)"',
        re.IGNORECASE
    )
    for m in pat_both.finditer(line):
        pairs.append((m.group(1), m.group(2)))

    # ★ 1) source_object_id ↔ source_object_name
    pat_src1 = re.compile(
        r'"source_object_id"\s*:\s*"([^"]+)"'
        r'.{0,400}?'
        r'"source_object_name"\s*:\s*"([^"]+)"',
        re.IGNORECASE
    )
    for m in pat_src1.finditer(line):
        pairs.append((m.group(1), m.group(2)))

    pat_src2 = re.compile(
        r'"source_object_name"\s*:\s*"([^"]+)"'
        r'.{0,400}?'
        r'"source_object_id"\s*:\s*"([^"]+)"',
        re.IGNORECASE
    )
    for m in pat_src2.finditer(line):
        # 순서가 반대라 group 순서에 유의
        pairs.append((m.group(2), m.group(1)))

    # ★ 2) target_object_id ↔ target_object_name
    pat_tgt1 = re.compile(
        r'"target_object_id"\s*:\s*"([^"]+)"'
        r'.{0,400}?'
        r'"target_object_name"\s*:\s*"([^"]+)"',
        re.IGNORECASE
    )
    for m in pat_tgt1.finditer(line):
        pairs.append((m.group(1), m.group(2)))

    pat_tgt2 = re.compile(
        r'"target_object_name"\s*:\s*"([^"]+)"'
        r'.{0,400}?'
        r'"target_object_id"\s*:\s*"([^"]+)"',
        re.IGNORECASE
    )
    for m in pat_tgt2.finditer(line):
        pairs.append((m.group(2), m.group(1)))

    # 3) 라벨만 있는 경우(기존)
    if not pairs:
        pat_lab = re.compile(
            rf'"{_LABEL_KEYS}"\s*:\s*"([^"]+)"',
            re.IGNORECASE
        )
        for m in pat_lab.finditer(line):
            pairs.append((None, m.group(1)))

    return pairs


def build_instance_index(lines: List[str]) -> List[Tuple[datetime, str, str]]:
    idx: List[Tuple[datetime, str, str]] = []
    seen = set()  # (ts, iid, lab) 중복 제거용

    for line in lines:
        dt = parse_line_time(line)
        if not dt:
            continue

        # 1) JSON 경로 (네 코드 그대로 유지)
        data = extract_json_from_line(line)
        if data:
            def collect_from_obj(obj: Any):
                if isinstance(obj, dict):
                    details = obj.get("details") if "details" in obj else obj
                    if isinstance(details, dict):
                        # 기존 후보들
                        inst_id = details.get("instanceId") or details.get("sharedId") or details.get("id")
                        label = (details.get("instanceLabel") or details.get("instance_label") or
                                details.get("label") or details.get("textDescription"))
                        if inst_id and label:
                            tup = (dt, str(inst_id), str(label))
                            if tup not in seen:
                                idx.append(tup); seen.add(tup)

                        # ★ 추가: 관계 이벤트 계열 (source/target 이름과 id)
                        # source
                        so_id = (details.get("source_object_id") or details.get("sourceId") or
                                details.get("source_id"))
                        so_name = (details.get("source_object_name") or details.get("sourceName") or
                                details.get("source_label"))
                        if so_id and so_name:
                            tup = (dt, str(so_id), str(so_name))
                            if tup not in seen:
                                idx.append(tup); seen.add(tup)

                        # target
                        ta_id = (details.get("target_object_id") or details.get("targetId") or
                                details.get("target_id"))
                        ta_name = (details.get("target_object_name") or details.get("targetName") or
                                details.get("target_label"))
                        if ta_id and ta_name:
                            tup = (dt, str(ta_id), str(ta_name))
                            if tup not in seen:
                                idx.append(tup); seen.add(tup)

                        # inputs.nodes[].data ... (기존 로직 유지)
                        if "inputs" in obj and isinstance(obj["inputs"], dict):
                            nodes = obj["inputs"].get("nodes")
                            if isinstance(nodes, list):
                                for n in nodes:
                                    if not isinstance(n, dict):
                                        continue
                                    data_n = n.get("data", {})
                                    inst_id2 = data_n.get("instanceId") or data_n.get("sharedId")
                                    label2 = (data_n.get("instanceLabel") or data_n.get("instance_label") or
                                            data_n.get("label") or data_n.get("textDescription"))
                                    if inst_id2 and label2:
                                        tup = (dt, str(inst_id2), str(label2))
                                        if tup not in seen:
                                            idx.append(tup); seen.add(tup)

                    # 재귀
                    for v in obj.values():
                        collect_from_obj(v)
                elif isinstance(obj, list):
                    for it in obj:
                        collect_from_obj(it)

            collect_from_obj(data)

        # 2) ★ JSON 성공/실패와 무관하게, 정규식 백업을 항상 돌림
        for iid, lab in extract_id_label_pairs_by_regex(line):
            if not lab:
                continue
            iid2 = str(iid) if iid else f"LBL:{lab}"
            tup = (dt, iid2, str(lab))
            if tup not in seen:
                idx.append(tup); seen.add(tup)

    idx.sort(key=lambda x: x[0])
    return idx

# def find_instance_id_by_label(idx: List[Tuple[datetime, str, str]], needle: str) -> Optional[str]:
#     """
#     1) 대소문자 구분해서 부분일치
#     2) 없으면 모두 소문자로 내려서 부분일치
#     3) 여러 개면 → 가장 최근(뒤쪽)의 인스턴스를 선택
#     """
#     needle_exact = needle
#     needle_lower = needle.lower()

#     exact_matches = [(ts, iid, lab) for (ts, iid, lab) in idx if needle_exact in lab]
#     if exact_matches:
#         return exact_matches[-1][1]

#     lower_matches = [(ts, iid, lab) for (ts, iid, lab) in idx if needle_lower in lab.lower()]
#     if lower_matches:
#         return lower_matches[-1][1]

#     return None

# ─────────────────────────────────────────────────────────────────────────────
# 3) 이벤트 생성 & 로그 합치기
# ─────────────────────────────────────────────────────────────────────────────

def make_created_event_line(abs_dt: datetime,
                            user_id: str,
                            src_id: str,
                            dst_id: str,
                            relation: str,
                            video_time_str: str) -> str:
    event_obj = {
        "timestamp": to_iso(abs_dt),
        "event": "relationship_node.created",
        "details": {
            "relationship_id": f"{src_id}->{dst_id}:{relation}",
            "sourceId": src_id,
            "targetId": dst_id,
            "relation": relation,
            "video_time": video_time_str
        },
        "platform": "backend",
        "user_id": user_id
    }
    return f"{to_log_prefix(abs_dt)} - INFO - {json.dumps(event_obj, ensure_ascii=False)}"

def merge_and_sort_lines(original: List[str], injected: List[str]) -> List[str]:
    # 원본은 (line, ts, orig_index), 삽입은 (line, ts, orig_index 큰 값부터)로 묶어 병합 정렬
    def pair(lines: List[str], base_idx_start: int = 0) -> List[Tuple[datetime, int, str]]:
        paired = []
        for i, line in enumerate(lines):
            ts = parse_line_time(line)
            # ts가 파싱 안 되면 아주 옛날/미래 값으로 밀지 않고, 그냥 None이면 뒤로 보내는 정책
            key_ts = ts if ts else datetime.max
            paired.append((key_ts, base_idx_start + i, line))
        return paired

    p1 = pair(original, 0)
    p2 = pair(injected, 10**9)  # tie-breaker로 원본을 우선
    merged = sorted(p1 + p2, key=lambda x: (x[0], x[1]))
    return [line for (_, _, line) in merged]

# ─────────────────────────────────────────────────────────────────────────────
# 4) 메인 루틴
# ─────────────────────────────────────────────────────────────────────────────

def main():
    base_dir = Path("./log/").resolve()
    summary_rows = []

    for user_id, conf in CONFIG.items():
        log_path = base_dir / f"Study2-{user_id}.log"
        if not log_path.exists():
            print(f"[WARN] {log_path.name} 없음 – 건너뜀")
            continue

        print(f"[INFO] 처리: {log_path.name}")
        lines = load_log_lines(log_path)
        idx = build_instance_index(lines)

        # 앵커 기준 계산
        anchor_vid_s = parse_video_time_to_seconds(conf["anchor_video_time"])
        anchor_log_dt = parse_iso_ts(conf["anchor_log_ts"])

        injected_lines: List[str] = []

        for (vid_time_str, src_label, dst_label, relation) in conf["events"]:
            # 이벤트 동영상 시각
            evt_vid_s = parse_video_time_to_seconds(vid_time_str)
            # 절대 로그 시각 = anchor_log_dt - anchor_vid + evt_vid
            abs_dt = anchor_log_dt - timedelta(seconds=anchor_vid_s) + timedelta(seconds=evt_vid_s)

            # 인스턴스 탐색
            src_id = find_instance_id_by_label(idx, src_label)
            dst_id = find_instance_id_by_label(idx, dst_label)

            # 매칭 실패 시, UNKNOWN으로 기록하되 메모 남김
            note = ""
            if src_id is None:
                note += f"[no src match: '{src_label}'] "
                src_id = f"UNKNOWN:{src_label}"
            if dst_id is None:
                note += f"[no dst match: '{dst_label}'] "
                dst_id = f"UNKNOWN:{dst_label}"

            # 이벤트 라인 생성
            line = make_created_event_line(
                abs_dt=abs_dt,
                user_id=user_id,
                src_id=src_id,
                dst_id=dst_id,
                relation=relation,
                video_time_str=vid_time_str
            )
            injected_lines.append(line)

            summary_rows.append({
                "user_id": user_id,
                "video_time": vid_time_str,
                "abs_log_ts": to_iso(abs_dt),
                "src_label": src_label,
                "dst_label": dst_label,
                "relation": relation,
                "src_instance_id": src_id,
                "dst_instance_id": dst_id,
                "note": note.strip()
            })

        # 병합 + 정렬 + 저장
        patched = merge_and_sort_lines(lines, injected_lines)
        out_path = base_dir / f"Study2-{user_id}.log.patched"
        out_path.write_text("\n".join(patched) + "\n", encoding="utf-8")
        print(f"[OK] 저장: {out_path.name} (삽입 {len(injected_lines)}줄)")

    # 요약 CSV 저장
    if summary_rows:
        import csv
        csv_path = base_dir / "inserted_relationship_events_summary-Study2.csv"
        with csv_path.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
            writer.writeheader()
            writer.writerows(summary_rows)
        print(f"[OK] 요약 저장: {csv_path.name}")

if __name__ == "__main__":
    main()


[INFO] 처리: Study2-P1.log
[OK] 저장: Study2-P1.log.patched (삽입 13줄)
[INFO] 처리: Study2-P3.log
[OK] 저장: Study2-P3.log.patched (삽입 6줄)
[INFO] 처리: Study2-P4.log
[OK] 저장: Study2-P4.log.patched (삽입 8줄)
[INFO] 처리: Study2-P5.log
[OK] 저장: Study2-P5.log.patched (삽입 9줄)
[INFO] 처리: Study2-P6.log
[OK] 저장: Study2-P6.log.patched (삽입 7줄)
[OK] 요약 저장: inserted_relationship_events_summary-Study2.csv
